In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 y el SDK de Qiskit

<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.3.0
```
</details>

El SDK de Qiskit proporciona algunas herramientas para convertir entre representaciones OpenQASM de programas cuánticos y la clase [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## Importar un programa OpenQASM 2 en Qiskit
Hay dos funciones para importar programas OpenQASM 2 en Qiskit.
Estas son [`qasm2.load()`](../api/qiskit/qasm2#load), que recibe un nombre de archivo, y [`qasm2.loads()`](../api/qiskit/qasm2#loads), que recibe el programa OpenQASM 2 como cadena de texto.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Consulta la [API de OpenQASM 2 de Qiskit](https://docs.quantum.ibm.com/api/qiskit/qasm2) para obtener más información.

### Importar programas simples
Para la mayoría de los programas OpenQASM 2, puedes usar `qasm2.load` y `qasm2.loads` con un solo argumento.

#### Ejemplo: importar un programa OpenQASM 2 como cadena de texto
Usa `qasm2.loads()` para importar un programa OpenQASM 2 como cadena de texto en un QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### Ejemplo: importar un programa OpenQASM 2 desde un archivo
Usa `load()` para importar un programa OpenQASM 2 desde un archivo en un QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### Vincular gates de OpenQASM 2 con gates de Qiskit
Por defecto, el importador de OpenQASM 2 de Qiskit trata el archivo de inclusión `"qelib1.inc"` como una biblioteca estándar *de facto*.
El importador trata este archivo como si contuviera exactamente los gates descritos en [el artículo original que define OpenQASM 2](https://arxiv.org/abs/1707.03429).
Qiskit usará los gates integrados de [la biblioteca de circuitos](../api/qiskit/circuit_library) para representar los gates en `"qelib1.inc"`.
Los gates definidos en el programa mediante sentencias `gate` de OpenQASM 2 se construirán, por defecto, como [subclases personalizadas de `Gate` de Qiskit](../api/qiskit/qiskit.circuit.Gate).

Puedes indicarle al importador que use clases [`Gate`](../api/qiskit/qiskit.circuit.Gate) específicas para las sentencias `gate` que encuentre.
También puedes usar este mecanismo para tratar nombres de gates adicionales como "integrados", es decir, que no requieran una definición explícita.
Si especificas qué clases de Gate usar para sentencias `gate` fuera de `"qelib1.inc"`, el circuito resultante será generalmente más eficiente de trabajar.

> **Warning:** A partir de la versión v1.0 del SDK de Qiskit, el *exportador* de OpenQASM 2 de Qiskit (consulta [Exportar un circuito de Qiskit a OpenQASM 2](#qasm2-export)) todavía se comporta como si `"qelib1.inc"` tuviera más gates de los que realmente tiene.
> Esto significa que la configuración predeterminada del importador podría no ser capaz de importar un programa exportado por nuestro exportador.
> Consulta [el ejemplo específico sobre cómo trabajar con el exportador heredado](#qasm2-import-legacy) para resolver este problema.
> 
> Esta discrepancia es un comportamiento heredado de Qiskit y [se resolverá en una versión posterior de Qiskit](https://github.com/Qiskit/qiskit/issues/10737).

Para pasar información sobre una instrucción personalizada al importador de OpenQASM 2, usa [la clase `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
Esta requiere cuatro datos en orden:

* El **nombre** del gate, tal como aparece en el programa OpenQASM 2
* El **número de parámetros angulares** que acepta el gate
* El **número de qubits** sobre los que actúa el gate
* La **clase o función constructora** de Python para el gate, que recibe los parámetros del gate (pero no los qubits) como argumentos individuales

Si el importador encuentra una definición `gate` que coincide con una instrucción personalizada dada, usará esa información personalizada para reconstruir el objeto gate.
Si se encuentra una sentencia `gate` que coincide con el `name` de una instrucción personalizada, pero no coincide con el número de parámetros ni con el número de qubits, el importador lanzará un [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) para indicar la discrepancia entre la información proporcionada y el programa.

Además, un quinto argumento `builtin` puede establecerse opcionalmente en `True` para hacer que el gate esté disponible automáticamente dentro del programa OpenQASM 2, incluso si no está definido explícitamente.
Si el importador encuentra una definición `gate` explícita para una instrucción personalizada integrada, la aceptará silenciosamente.
Como antes, si una definición explícita del mismo nombre no es compatible con la instrucción personalizada proporcionada, se lanzará un [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror).
Esto es útil para la compatibilidad con exportadores de OpenQASM 2 más antiguos y con ciertas plataformas cuánticas que tratan los "gates base" de su hardware como instrucciones integradas.

Qiskit proporciona un atributo de datos para trabajar con programas OpenQASM 2 producidos por versiones antiguas de [las capacidades de exportación de OpenQASM 2 de Qiskit](#qasm2-export).
Este es [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), que puede pasarse como argumento `custom_instructions` a [`qasm2.load()`](../api/qiskit/qasm2#load) y [`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### Ejemplo: importar un programa creado por el exportador heredado de Qiskit
Este programa OpenQASM 2 utiliza gates que no están en la versión original de `"qelib1.inc"` sin declararlos, pero son gates estándar en la biblioteca de Qiskit.
Puedes usar [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) para indicarle fácilmente al importador que use el mismo conjunto de gates que el exportador de OpenQASM 2 de Qiskit usaba anteriormente.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### Ejemplo: usar una clase de gate particular al importar un programa OpenQASM 2
En general, Qiskit no puede verificar si la definición en una sentencia `gate` de OpenQASM 2 corresponde exactamente a un gate de la biblioteca estándar de Qiskit.
En cambio, Qiskit elige un gate personalizado usando la definición exacta proporcionada.
Esto puede ser menos eficiente que usar uno de los gates estándar integrados o un gate personalizado definido por el usuario.
Puedes definir manualmente sentencias `gate` con clases particulares.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### Ejemplo: definir un nuevo gate integrado en un programa OpenQASM 2
Si se establece el argumento `builtin=True`, un gate personalizado no necesita tener una definición asociada.